# Aula 3 - Composição da desocupação por sexo (2012 x 2026) cruzada com afazeres domésticos

Nesta aula:
1. Abrimos e juntamos as duas **Tabelas 5** (composição da desocupação por sexo, 2012 e 2026)
2. Comparamos ano a ano (variação em pontos percentuais)
3. Cruzamos com a **Tabela 1.1.1** do IBGE (horas semanais de cuidados/afazeres domésticos, 2022), tratada do mesmo jeito que na Aula 2


In [1]:
import pandas as pd

## 1) Ler e juntar as duas Tabelas 5 (2012 e 2026)

In [2]:
t2012 = pd.read_csv(
    "Tabela5-sem_emprego_2012.csv",
    sep=";", decimal=",", encoding="utf-8-sig"
)
t2026 = pd.read_csv(
    "Tabela5-sem_emprego_2026.csv",
    sep=";", decimal=",", encoding="utf-8-sig"
)

comp = t2012.merge(t2026, on=["Sigla", "Código", "Estado"], how="inner")
comp.head()

,Sigla,Código,Estado,Desocupados - homens (2012 T1),Desocupados - mulheres (2012 T1),Desocupados - homens (2026 T1),Desocupados - mulheres (2026 T1)
0,AC,12,Acre,45.3,54.7,53.2,46.8
1,AL,27,Alagoas,44.7,55.3,50.7,49.3
2,AM,13,Amazonas,47.5,52.5,46.8,53.2
3,AP,16,Amapá,48.1,51.9,51.1,48.9
4,BA,29,Bahia,43.8,56.2,44.7,55.3


## 2) Comparar ano a ano (variação 2026 - 2012, em pontos percentuais)

In [3]:
comp["Var. homens (p.p.)"] = (
    comp["Desocupados - homens (2026 T1)"] - comp["Desocupados - homens (2012 T1)"]
).round(1)
comp["Var. mulheres (p.p.)"] = (
    comp["Desocupados - mulheres (2026 T1)"] - comp["Desocupados - mulheres (2012 T1)"]
).round(1)

comp = comp.sort_values("Var. mulheres (p.p.)")
comp

,Sigla,Código,Estado,Desocupados - homens (2012 T1),Desocupados - mulheres (2012 T1),Desocupados - homens (2026 T1),Desocupados - mulheres (2026 T1),Var. homens (p.p.),Var. mulheres (p.p.)
0,AC,12,Acre,45.3,54.7,53.2,46.8,7.9,-7.9
6,DF,53,Distrito Federal,41.0,59.0,47.9,52.1,6.9,-6.9
8,GO,52,Goiás,41.0,59.0,47.5,52.5,6.5,-6.5
1,AL,27,Alagoas,44.7,55.3,50.7,49.3,6.0,-6.0
17,PR,41,Paraná,40.3,59.7,46.3,53.7,6.0,-6.0
10,MG,31,Minas Gerais,43.1,56.9,48.6,51.4,5.5,-5.5
13,PA,15,Pará,43.2,56.8,48.3,51.7,5.1,-5.1
15,PE,26,Pernambuco,43.9,56.1,48.7,51.3,4.8,-4.8
22,RS,43,Rio Grande do Sul,42.8,57.2,47.4,52.6,4.6,-4.6
14,PB,25,Paraíba,44.4,55.6,48.9,51.1,4.5,-4.5


## 3) Ler a Tabela 1.1.1 do IBGE (aba 2022) — mesmo tratamento da Aula 2

- pula as 8 primeiras linhas (título + cabeçalho em várias linhas)
- nomeia as colunas na mão
- remove linhas de Grandes Regiões / Brasil / rodapé de fonte e notas

In [4]:
colunas = [
    "Estado",
    "Horas - total",
    "Horas - total (brancos)",
    "Horas - total (pretos/pardos)",
    "Horas - homens (brancos)",
    "Horas - homens (pretos/pardos)",
    "Horas - mulheres (brancas)",
    "Horas - mulheres (pretas/pardas)",
]

tab111 = pd.read_excel(
    "Tabela_1_1_1__1_.xls",
    sheet_name="2022",
    header=None,
    skiprows=8,
    names=colunas,
)

grandes_regioes = ["Brasil", "Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]
tab111 = tab111.dropna(subset=["Estado"])
tab111 = tab111[~tab111["Estado"].isin(grandes_regioes)]
tab111 = tab111[pd.to_numeric(tab111["Horas - total"], errors="coerce").notna()]

# médias simples homens / mulheres (branco + preto ou pardo)
tab111["Horas - homens"] = tab111[["Horas - homens (brancos)", "Horas - homens (pretos/pardos)"]].mean(axis=1).round(2)
tab111["Horas - mulheres"] = tab111[["Horas - mulheres (brancas)", "Horas - mulheres (pretas/pardas)"]].mean(axis=1).round(2)
tab111["Horas - total"] = tab111["Horas - total"].round(2)

tab111.head()

,Estado,Horas - total,Horas - total (brancos),Horas - total (pretos/pardos),Horas - homens (brancos),Horas - homens (pretos/pardos),Horas - mulheres (brancas),Horas - mulheres (pretas/pardas),Horas - homens,Horas - mulheres
2,Rondônia,17.01,16.727733,17.101523,12.111127,12.544080,20.412887,21.061410,12.33,20.74
3,Acre,13.61,13.806929,13.541399,10.796120,9.686983,16.130352,16.710754,10.24,16.42
4,Amazonas,16.81,15.987545,16.995493,12.233043,12.822584,18.764969,20.716986,12.53,19.74
5,Roraima,13.64,13.651755,13.633606,10.814803,10.476601,16.135877,16.560244,10.65,16.35
6,Pará,16.47,16.181627,16.560239,11.524306,11.026707,19.835110,21.591144,11.28,20.71


## 4) Dar Sigla/Código à Tabela 1.1.1

A Tabela 1.1.1 só traz o nome do estado. Usamos o de-para que já existe em `comp` (mesmos nomes de Estado das Tabelas 5) para trazer Sigla e Código.

In [5]:
de_para = comp[["Sigla", "Código", "Estado"]]
tab111 = de_para.merge(tab111, on="Estado", how="inner")
tab111.head()

,Sigla,Código,Estado,Horas - total,Horas - total (brancos),Horas - total (pretos/pardos),Horas - homens (brancos),Horas - homens (pretos/pardos),Horas - mulheres (brancas),Horas - mulheres (pretas/pardas),Horas - homens,Horas - mulheres
0,AC,12,Acre,13.61,13.806929,13.541399,10.796120,9.686983,16.130352,16.710754,10.24,16.42
1,DF,53,Distrito Federal,15.10,15.061306,15.126295,11.377952,10.481279,17.738123,19.307332,10.93,18.52
2,GO,52,Goiás,15.19,14.962622,15.296966,10.636424,10.531883,18.440842,19.587913,10.58,19.01
3,AL,27,Alagoas,20.22,19.165111,20.560451,12.121839,13.357160,24.093655,25.856637,12.74,24.98
4,PR,41,Paraná,15.39,15.453295,15.307289,11.240085,10.815611,18.896661,19.365380,11.03,19.13


## 5) Cruzar a comparação de desocupação com a Tabela 1.1.1

In [6]:
cruzado = comp.merge(tab111, on=["Sigla", "Código", "Estado"], how="inner")
cruzado

,Sigla,Código,Estado,Desocupados - homens (2012 T1),Desocupados - mulheres (2012 T1),Desocupados - homens (2026 T1),Desocupados - mulheres (2026 T1),Var. homens (p.p.),Var. mulheres (p.p.),Horas - total,Horas - total (brancos),Horas - total (pretos/pardos),Horas - homens (brancos),Horas - homens (pretos/pardos),Horas - mulheres (brancas),Horas - mulheres (pretas/pardas),Horas - homens,Horas - mulheres
0,AC,12,Acre,45.3,54.7,53.2,46.8,7.9,-7.9,13.61,13.806929,13.541399,10.796120,9.686983,16.130352,16.710754,10.24,16.42
1,DF,53,Distrito Federal,41.0,59.0,47.9,52.1,6.9,-6.9,15.10,15.061306,15.126295,11.377952,10.481279,17.738123,19.307332,10.93,18.52
2,GO,52,Goiás,41.0,59.0,47.5,52.5,6.5,-6.5,15.19,14.962622,15.296966,10.636424,10.531883,18.440842,19.587913,10.58,19.01
3,AL,27,Alagoas,44.7,55.3,50.7,49.3,6.0,-6.0,20.22,19.165111,20.560451,12.121839,13.357160,24.093655,25.856637,12.74,24.98
4,PR,41,Paraná,40.3,59.7,46.3,53.7,6.0,-6.0,15.39,15.453295,15.307289,11.240085,10.815611,18.896661,19.365380,11.03,19.13
5,MG,31,Minas Gerais,43.1,56.9,48.6,51.4,5.5,-5.5,17.50,17.179387,17.707564,11.319720,11.619120,22.186650,23.075751,11.47,22.63
6,PA,15,Pará,43.2,56.8,48.3,51.7,5.1,-5.1,16.47,16.181627,16.560239,11.524306,11.026707,19.835110,21.591144,11.28,20.71
7,PE,26,Pernambuco,43.9,56.1,48.7,51.3,4.8,-4.8,19.31,18.950219,19.418444,12.312833,12.868821,23.357880,24.243396,12.59,23.80
8,RS,43,Rio Grande do Sul,42.8,57.2,47.4,52.6,4.6,-4.6,15.36,15.337269,15.448203,11.308447,11.519941,18.868230,19.107720,11.41,18.99
9,PB,25,Paraíba,44.4,55.6,48.9,51.1,4.5,-4.5,18.92,18.887920,18.945306,12.480204,12.400831,23.642936,24.035650,12.44,23.84


## 6) Salvar resultados

No seu PC, troque o caminho por algo como `r"H:\dados30\python-tecTI\dados\geral.csv"`.

In [7]:
comp.to_csv("comp_2012_2026.csv", sep=";", decimal=",", index=False)
cruzado.to_csv("geral.csv", sep=";", decimal=",", index=False)

## 7) Alguns achados rápidos

In [8]:
print("Média nacional 2012 - homens/mulheres:", comp["Desocupados - homens (2012 T1)"].mean().round(1), comp["Desocupados - mulheres (2012 T1)"].mean().round(1))
print("Média nacional 2026 - homens/mulheres:", comp["Desocupados - homens (2026 T1)"].mean().round(1), comp["Desocupados - mulheres (2026 T1)"].mean().round(1))
print()
print("Correlação Var. mulheres (p.p.) x Horas - mulheres:", cruzado["Var. mulheres (p.p.)"].corr(cruzado["Horas - mulheres"]).round(3))

Média nacional 2012 - homens/mulheres: 45.4 54.6
Média nacional 2026 - homens/mulheres: 47.0 53.0

Correlação Var. mulheres (p.p.) x Horas - mulheres: 0.103
